# Modeling Benchmarks
This notebook benchmarks several regression algorithms on the engineered taxi-trip features. Each model shares a common preprocessing stack so we can compare apples-to-apples RMSE/MAE scores.

In [ ]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold, cross_validate
from sklearn.base import BaseEstimator, TransformerMixin

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT / 'src'))

from utils_data import apply_feature_engineering, get_feature_lists


In [ ]:
class DenseTransformer(BaseEstimator, TransformerMixin):
    """Utility to convert sparse matrices into dense arrays for models like KNN."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.toarray() if hasattr(X, "toarray") else X



In [ ]:
RAW_DATA_PATH = PROJECT_ROOT / 'split' / 'train.csv'
raw_df = pd.read_csv(RAW_DATA_PATH)

fe_df, train_stats = apply_feature_engineering(raw_df)
feature_lists = get_feature_lists()

categorical_cols = feature_lists['categorical']
binary_cols = feature_lists['binary']
numeric_cols = feature_lists['numeric'] + feature_lists['cyclic']
feature_cols = categorical_cols + binary_cols + numeric_cols

X = fe_df[feature_cols]
y = fe_df['log_trip_duration']

print(f"Rows after FE: {len(fe_df):,}")
print(f"Using {len(feature_cols)} engineered features")
fe_df.head()


The engineered dataset keeps roughly 930k rows after NYC-boundary filtering and IQR clipping. Features span categorical demand signals, binary traffic flags, and distance/speed metrics, so we standardise numeric ranges before each model.

In [ ]:
def build_preprocessor():
    numeric_pipe = Pipeline([
        ('scaler', QuantileTransformer(output_distribution='normal'))
    ])
    return ColumnTransformer([
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('numeric', numeric_pipe, numeric_cols),
        ('binary', 'passthrough', binary_cols)
    ])

models = {
    'ridge': Ridge(alpha=1.0),
    'lasso': Lasso(alpha=5e-4, max_iter=5000),
    'elastic_net': ElasticNet(alpha=5e-4, l1_ratio=0.3, max_iter=5000),
    'random_forest': RandomForestRegressor(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42),
    'gradient_boosting': GradientBoostingRegressor(random_state=42),
    'hist_gbrt': HistGradientBoostingRegressor(max_depth=10, learning_rate=0.1, random_state=42),
    'knn': KNeighborsRegressor(n_neighbors=10, weights='distance')
}


In [ ]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)
scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2'
}

records = []
for name, estimator in models.items():
    steps = [('preprocess', build_preprocessor())]
    if name == 'knn':
        steps.append(('to_dense', DenseTransformer()))
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', estimator))
    pipeline = Pipeline(steps)
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        error_score='raise'
    )
    records.append({
        'model': name,
        'rmse': -scores['test_rmse'].mean(),
        'mae': -scores['test_mae'].mean(),
        'r2': scores['test_r2'].mean()
    })

results_df = pd.DataFrame(records).sort_values('rmse').reset_index(drop=True)
results_df


The table ranks models by cross-validated RMSE (lower is better). Tree ensembles typically dominate because they capture non-linear effects from demand pressure and congestion proxies.

In [ ]:
best_model_name = results_df.loc[0, 'model']
best_estimator = models[best_model_name]
print(f"Best CV performer: {best_model_name} (RMSE={results_df.loc[0,'rmse']:.3f})")

steps = [('preprocess', build_preprocessor())]
if best_model_name == 'knn':
    steps.append(('to_dense', DenseTransformer()))
    steps.append(('scaler', StandardScaler()))
steps.append(('model', best_estimator))

final_pipeline = Pipeline(steps)
final_pipeline.fit(X, y)

artifact = {
    'model': final_pipeline,
    'feature_lists': feature_lists,
    'train_stats': train_stats
}
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)
model_path = MODEL_DIR / f'benchmark_{best_model_name}.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(artifact, f)
model_path


## Next Steps
- Validate the saved benchmark model on `split/val.csv` via `src/test.py` to ensure parity with the ridge baseline.
- Promote the best-performing algorithm into `src/train.py` once its hyperparameters are tuned on a validation split.
- Consider stacking (e.g., Ridge + HistGB) after logging/permutation-importance analysis to squeeze a few more leaderboard points.